## Why Train/Test Split Is Not Enough

### Why can a single split give misleading results?
- Single split can give misleading result because the performance estimate depends heavily on how the data happens to be divided. If the split is not representative—due to randomness, class imbalance, or temporal/ordering patterns—the model may appear to perform well on one split but poorly on another, leading to unreliable conclusions. 

### What happens if the split is unusually easy or unusually difficult?
- In both scenarios increase variance in evaluation and reduce reliability.
- In case of easy split test set gets simple and well related instances which leads to overly optimistic performance metric but model fails in real-world.
- In case of difficult split test gets rare and complex instances which leads to bad performance metric thus making model seem worse then it actually is.

### How could model evaluation change with a different random seed?
- If random seed is different then train-test split would be different this might lead to slightly different model's evaluation metrics.


## Setting Baseline

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("cleaned_data.csv")

x = df.drop("default_payment_next_month", axis=1)
y = df["default_payment_next_month"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

numeric_feature = [
    "limit_bal", "age", "bill_amt1", "bill_amt2", "bill_amt3", "bill_amt4", "bill_amt5", "bill_amt6", "pay_amt1", "pay_amt2", "pay_amt3", "pay_amt4", "pay_amt5", "pay_amt6"
]

preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), numeric_feature)], remainder="passthrough")

In [21]:
neg, pos = y_train.value_counts()
scale_pos_weight = neg/pos
scale_pos_weight

3.519607843137255

In [22]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb_weighted = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=400,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
            scale_pos_weight=scale_pos_weight
        ))
    ]
)

xgb_weighted.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [23]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score, precision_score, f1_score

y_pred = xgb_weighted.predict(x_test)
y_prob = xgb_weighted.predict_proba(x_test)[ : ,1]

print("                    : Weighted : \n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix : \n", confusion_matrix(y_test, y_pred))
print("ROC_AUC :", roc_auc_score(y_test, y_prob))
print("Recall :", recall_score(y_test, y_pred))
print("Precision Score :", precision_score(y_test, y_pred))
print("f1_score :", f1_score(y_test, y_pred))

                    : Weighted : 

              precision    recall  f1-score   support

           0       0.88      0.79      0.83      4667
           1       0.46      0.63      0.53      1326

    accuracy                           0.76      5993
   macro avg       0.67      0.71      0.68      5993
weighted avg       0.79      0.76      0.77      5993

Confusion Matrix : 
 [[3696  971]
 [ 492  834]]
ROC_AUC : 0.7729232979803318
Recall : 0.6289592760180995
Precision Score : 0.46204986149584487
f1_score : 0.5327371446822101


## K-Fold Cross-Validation

### What is cross-validation?
- Cross-validation is a model evaluation technique used to obtain a more reliable estimate of model performance. Instead of evaluating the model using a single train–test split, the training dataset is divided into k subsets (folds). The model is then trained k times, where in each iteration one fold is used as the validation set and the remaining folds are used for training. The performance scores from all iterations are then averaged to estimate the model’s overall performance.

In [24]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    xgb_weighted,
    x_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)

scores, scores.std()

(array([0.78246664, 0.77307842, 0.7792919 , 0.77927484, 0.78687751]),
 np.float64(0.004519386107174787))

### What does each fold represent?
- Fold represents one of the k equal sized subset in which training dataset is divided.

### Why does averaging across folds give a more stable estimate?
- Averaging across folds gives a more stable estimate because the model is trained and evaluated multiple times on different training and validation splits. This reduces the impact of randomness from a single data split and provides a more reliable estimate of how the model will perform on unseen data.

### How different are the fold scores?
- The fold scores are not significantly different from each other, which indicates that the model performs consistently across different subsets of the dataset. This suggests that the model is reasonably stable and not overly sensitive to a particular data split.

## Stratified Cross-Validation

### What is Stratified Cross-Validation?
- Stratified cross-validation is a variation of K-Fold cross-validation designed specifically for classification problems, especially when the dataset is imbalanced. Each fold preserves the same class distribution as the full dataset.

In [25]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    xgb_weighted,
    x_train,
    y_train,
    cv=skf,
    scoring='roc_auc'
)

scores, scores.std()

(array([0.78540724, 0.78605178, 0.77225925, 0.76959754, 0.79254417]),
 np.float64(0.008769047147489427))

### Why is stratification important in imbalanced datasets?
- Because it preserves same class distribution as the full dataset thus training and validating each times on split with similar proportion of class as full which gives better evaluation metrics.

### What could happen if a fold contains very few positive cases?
- if fold contains very few positive cases then model will not learn to identify positive cases thus decreasing it's evaluation metrics and overall performance of model.

## Cross-Validation With Pipelines

- When using cross-validation, preprocessing must be inside a pipeline. 
- If scaling is performed before cross-validation, the scaler is fit on the entire dataset, which leaks information from validation folds into training folds. 

- Pipelines prevent this by fitting preprocessing only on the training portion of each fold and then applying the transformation to the validation fold separately.

## Leakage Detection Exercise

### Could any feature contain future information?
- Yes, if the feature was recorded after class (target) has been assigned then that feature contains information of future.
- Future information means a feature that would not be available at prediction time.

### Could engineered features accidentally leak the target?
- Yes. This happens frequently when creating aggregated or derived features.

## Hyperparameter Tuning With Nested Validation

- Nested validation (or nested cross-validation) is a technique used to obtain an unbiased estimate of model performance when hyperparameters are being tuned.

In [26]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators" : [400, 500],
    "classifier__max_depth": [3, 5],
    # "classifier__learning_rate": [0.01, 0.1],
    "classifier__subsample": [0.8],
    "classifier__colsample_bytree": [0.8]
}

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    xgb_weighted,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="roc_auc",
)

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

nested_scores = cross_val_score(
    grid_search,
    x_train,
    y_train,
    cv=outer_cv,
    scoring="roc_auc"
)

nested_scores

array([0.78540724, 0.78605178, 0.77225925, 0.76959754, 0.79254417])

### Why does tuning inside the same CV inflate scores?
- Hyperparameter tuning inside the same cross-validation inflates scores because the validation folds are used both to select the best hyperparameters and to estimate model performance. Since the model is chosen based on its performance on those folds, it becomes indirectly optimized for them, leading to optimistic bias. Nested cross-validation avoids this by separating hyperparameter tuning (inner CV) from performance evaluation (outer CV).

### When is nested CV necessary?
- Nested cross-validation is necessary when we need an unbiased estimate of model performance while performing hyperparameter tuning, especially with small datasets or large hyperparameter search spaces. It separates model selection (inner CV) from performance evaluation (outer CV). In workflows where a separate test set is available and only the training data is used for cross-validation during tuning, nested CV is usually not required.

## Metric Stability Analysis

- Metric stability can be evaluated by examining the distribution of cross-validation scores. The mean score provides the expected performance, while the standard deviation indicates how sensitive the model is to different data splits. A small standard deviation suggests stable performance, whereas a large variance may indicate model instability or dataset heterogeneity.

In [27]:
from sklearn.model_selection import RepeatedKFold

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

scores = cross_val_score(
    xgb_weighted,
    x_train,
    y_train,
    cv=rkf,
    scoring="roc_auc"
)

print("ROC-AUC Score :", scores)
print("Mean :", scores.mean())
print("STD :", scores.std())

ROC-AUC Score : [0.77950378 0.78712494 0.77046307 0.78117829 0.77344742 0.78794609
 0.77070778 0.7834151  0.78969128 0.77359523 0.77720942 0.77106883
 0.78993553 0.77464786 0.78650695]
Mean : 0.7797627712580107
STD : 0.007018019212149263


### How stable is ROC-AUC across repetitions?
- ROC-AUC is very stable across repetitions as variation is only 0.7%.

## Validation vs Test Set

### What is the purpose of the validation set?
- The validation set is used during model development to evaluate different model configurations, tune hyperparameters, and guide feature engineering decisions. It provides feedback on how well the model generalizes to unseen data without using the test set. This prevents overfitting to the test set and ensures that the final evaluation remains unbiased.

### Why should the test set be used only once?
- The primary purpose of the test set is to provide an unbiased evaluation of a fully trained and tuned machine learning model's performance on completely unseen data.  It simulates real-world conditions by acting as a final checkpoint after all training and hyperparameter tuning are complete.

### Why should the test set be used only once?
- The test set should be used only once because it is meant to provide an unbiased estimate of how the model performs on completely unseen data.

## Final Model Evaluation Protocol

- Final Model Evaluation Protocol:

1. Split the dataset into training and test sets.
2. Perform cross-validation on the training data to evaluate model performance.
3. Tune hyperparameters using grid search within cross-validation.
4. Retrain the selected model on the full training dataset.
5. Evaluate the final model once on the held-out test set to obtain an unbiased estimate of performance.